## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Capability contract

**Capability:** C03 — Correctness Contracts & Idempotency  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Detect a correctness failure


## 🖊️ Sketch note

![Sketch note — API → Parquet → DuckDB](../assets/sketch-notes/11-api-parquet-duckdb.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S02 — API schema drift (DATA-1051)** → [`../work_simulations/02_acme_commerce_api_schema_drift/README.md`](../work_simulations/02_acme_commerce_api_schema_drift/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Measure

Record an observable result that lets you compare your prediction with what actually happened.


## Learning objectives

- validate an external-style input;
- normalize it into a stable schema;
- publish a queryable artifact;
- measure and explain one quality property.


## 1. Mental model

```text
external source
      ↓
   raw boundary       ← preserve what arrived
      ↓
   validation
      ↓
 normalization
      ↓
 curated Parquet     ← durable analytical artifact
      ↓
    DuckDB            ← compute close to the data
      ↓
 analytical question
```

The important boundary is **raw vs curated**. Raw data helps replay and diagnose. Curated data gives downstream consumers a stable contract.

## 2. Predict before you run

> If one record is missing `title`, should the pipeline silently write a null into the curated dataset, or fail before publication? Why?

Write your answer before executing the next cell.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"duckdb": "duckdb>=1.0,<2", "pyarrow": "pyarrow>=16,<22", "pandas": "pandas>=2,<4", "requests": "requests>=2.31,<3"}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from pathlib import Path
import json, time
import requests
import pandas as pd
import duckdb

WORK = Path("data/11_api_parquet_duckdb")
WORK.mkdir(parents=True, exist_ok=True)
RAW = WORK / "raw_posts.json"
CURATED = WORK / "curated_posts.parquet"

FIXTURE = [
    {"userId": 1, "id": 101, "title": "Reliable ingestion", "body": "Validate at the boundary."},
    {"userId": 2, "id": 102, "title": "Parquet as a data product", "body": "Persist a stable analytical shape."},
    {"userId": 1, "id": 103, "title": "Query close to the data", "body": "DuckDB can query local files."},
]

API_URL = "https://jsonplaceholder.typicode.com/posts"
try:
    r = requests.get(API_URL, timeout=8)
    r.raise_for_status()
    records = r.json()[:30]
    source = "live API"
except Exception as exc:
    print(f"Live API unavailable ({type(exc).__name__}); using deterministic fixture.")
    records = FIXTURE
    source = "local fixture"

RAW.write_text(json.dumps(records, indent=2), encoding="utf-8")
print(f"Source: {source}; records: {len(records)}; raw bytes: {RAW.stat().st_size}")

## 3. Validate the source contract

A contract is useful only when the pipeline actually enforces it. Keep the first contract intentionally small.

In [ ]:
REQUIRED = {"userId", "id", "title", "body"}

def validate_post(record: dict) -> None:
    missing = REQUIRED - record.keys()
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")
    if not isinstance(record["id"], int):
        raise TypeError("id must be an integer")
    if not isinstance(record["userId"], int):
        raise TypeError("userId must be an integer")
    if not isinstance(record["title"], str) or not record["title"].strip():
        raise ValueError("title must be a non-empty string")
    if not isinstance(record["body"], str):
        raise TypeError("body must be a string")

for record in records:
    validate_post(record)
print("Contract checks passed for every source record.")

## 4. Normalize and publish curated Parquet

Normalization creates a downstream-friendly schema. We also add an explicit `ingested_at` so the dataset carries pipeline metadata instead of relying on filesystem timestamps.

In [ ]:
ingested_at = pd.Timestamp.now(tz="UTC").isoformat()
curated = pd.DataFrame([
    {
        "post_id": r["id"],
        "user_id": r["userId"],
        "title": r["title"].strip(),
        "body": r["body"].strip(),
        "ingested_at": ingested_at,
    }
    for r in records
])

assert curated["post_id"].is_unique
assert curated["title"].notna().all()
curated.to_parquet(CURATED, index=False)
print(curated.head(5).to_string(index=False))
print(f"\nCurated rows: {len(curated)}; Parquet bytes: {CURATED.stat().st_size}")

## 5. Query Parquet with DuckDB

DuckDB lets us ask analytical SQL questions without first loading the entire file into a database server.

In [ ]:
con = duckdb.connect()
summary = con.execute(f"""
    SELECT user_id,
           COUNT(*) AS posts,
           AVG(LENGTH(body)) AS avg_body_chars
    FROM read_parquet('{CURATED.as_posix()}')
    GROUP BY user_id
    ORDER BY user_id
""").fetchdf()
display(summary)

plan = con.execute(f"EXPLAIN SELECT COUNT(*) FROM read_parquet('{CURATED.as_posix()}')").fetchall()
print("\nQuery plan excerpt:")
print("\n".join(str(row[1] if len(row)>1 else row[0]) for row in plan)[:1200])

## 6. Measure

Measure the artifact and a representative query. Do not generalize a tiny benchmark into a universal performance claim.

In [ ]:
query = f"SELECT user_id, COUNT(*) AS posts FROM read_parquet('{CURATED.as_posix()}') GROUP BY user_id"
t0 = time.perf_counter()
for _ in range(5):
    con.execute(query).fetchall()
elapsed = (time.perf_counter() - t0) / 5

print(f"Raw JSON size:      {RAW.stat().st_size:,} bytes")
print(f"Curated Parquet:    {CURATED.stat().st_size:,} bytes")
print(f"Average query time: {elapsed*1000:.2f} ms")
print("These numbers describe this tiny local experiment, not all workloads.")

## 💥 7. Break it deliberately

Simulate a source contract change. The point is to observe **where** the pipeline should fail.

In [ ]:
broken = dict(records[0])
broken.pop("title")

try:
    validate_post(broken)
except Exception as exc:
    print("Controlled failure:", type(exc).__name__, "→", exc)
else:
    raise AssertionError("The broken record should have failed validation")

## 8. Diagnose → Improve

The failure occurs before Parquet publication. That is desirable: downstream consumers never receive a dataset that violates the curated contract.

A useful production pattern is therefore:

**fetch → preserve raw → validate → normalize → atomically publish curated artifact**

For a larger system, publishing also needs versioning, metadata, retries, access control and lineage.

In [ ]:
# A small publication guard: write to a temporary artifact, validate it, then promote it.
TEMP = CURATED.with_suffix(".tmp.parquet")
curated.to_parquet(TEMP, index=False)
check = pd.read_parquet(TEMP)
assert check["post_id"].is_unique
assert check["title"].notna().all()
TEMP.replace(CURATED)
print("Curated artifact passed publication checks and was promoted atomically.")

## 9. What changes in production?

At 1,000× the data, the architecture needs to consider partitioning, object storage, parallel ingestion and query pruning. If the API fails, retry with bounded backoff and preserve the failed response/error metadata. If a schema changes, contract checks should stop publication and alert an operator. The raw layer should remain replayable.

**Technology bridge:** object storage + Parquet is common for durable analytical data; DuckDB is useful for local/serverless analytical work; larger teams may add a catalog, orchestration and a warehouse/lakehouse engine.

## 10. Explain it in 60–90 seconds

> Explain why “API → Parquet” is not one operation. Where are the contracts, state boundaries and failure points?

## 11. Challenge

Extend the pipeline so it:
1. rejects duplicate `post_id` values;
2. records a small run manifest with source, row count, schema and timestamp;
3. answers: “Which user has the longest average post body?”

Then deliberately break one of those assumptions and make the failure observable.

## Takeaways

- Raw and curated data serve different recovery and consumption needs.
- Validation is a publication gate, not a documentation exercise.
- Parquet gives a durable columnar artifact; DuckDB can query it directly.
- Small measured experiments are evidence for this workload, not universal benchmarks.
- A production pipeline must make failure, replay and ownership explicit.